In [ ]:
#Step 1 - INSTALL LIBRARIES
!pip install requests pandas openpyxl geopy

In [ ]:
#Step 2 - GET CODE
#replace client_id=xxxxxx with your CLIENT_ID
https://www.strava.com/oauth/authorize?client_id=xxxxxx&response_type=code&redirect_uri=http://localhost&approval_prompt=force&scope=activity:read_all

In [ ]:
#Step 3 - GET ACCESS_TOKEN with the CODE

import requests

CLIENT_ID = "MY_CLIENT_ID" # replace with your CLIENT_ID
CLIENT_SECRET = "MY_CLIENT_SECRET" # replace with your CLIENT_SECRET
CODE = "MY_CODE" # replace with your CODE

response = requests.post(
    "https://www.strava.com/oauth/token",
    data={
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "code": CODE,
        "grant_type": "authorization_code"
    }
)

print(response.json())


In [ ]:
#STEP 4 - FETCH STRAVA ACTIVITIES and SHOE INFO

import requests
import pandas as pd
import math
import time
from geopy.geocoders import Nominatim
from datetime import datetime

#start time
print(f'Process started at {datetime.now()}')

# -------------------------
# CONFIGURATION
# -------------------------
ACCESS_TOKEN = "MY_TOKEN"  # replace with your access token
PER_PAGE = 200  # max per Strava API
PAGE = 1
headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
FOLDER_TO_SAVE = r'C:\Users\xxxxx\Documents\Strava Data' #replace with your FOLDER to save 
FILE_NAME = FOLDER_TO_SAVE + '\\' + 'strava_database.xlsx' #replace with your preferred file name

# -------------------------
# GEOCODER SETUP
# -------------------------
geolocator = Nominatim(user_agent="strava_activity_export")
location_cache = {}

def reverse_lookup(lat, lon):
    # Round to reduce duplicate lookups
    key = (round(lat, 3), round(lon, 3))
    
    if key in location_cache:
        return location_cache[key]
    
    try:
        location = geolocator.reverse(f"{lat}, {lon}", language="en")
        if location and "address" in location.raw:
            address = location.raw["address"]
            city = (
                address.get("city")
                or address.get("town")
                or address.get("village")
                or address.get("hamlet")
            )
            country = address.get("country")
        else:
            city, country = None, None
    except Exception:
        city, country = None, None
    
    location_cache[key] = (city, country)
    
    time.sleep(1)
    return city, country

# -------------------------
# HELPER FUNCTION: Calculate pace
# -------------------------
def calculate_pace(distance_m, moving_time_s):
    distance_km = distance_m / 1000
    moving_time_min = moving_time_s / 60
    if distance_km > 0:
        pace_min_per_km = moving_time_min / distance_km
        return round(pace_min_per_km, 2)
    return None

def format_pace(pace_min_per_km):
    if pace_min_per_km:
        minutes = int(math.floor(pace_min_per_km))
        seconds = int(round((pace_min_per_km - minutes) * 60))
        return f"{minutes}:{seconds:02d} / km"
    return None

# -------------------------
# FETCH ACTIVITIES
# -------------------------
activities = []
while True:
    response = requests.get(
        "https://www.strava.com/api/v3/athlete/activities",
        headers={"Authorization": f"Bearer {ACCESS_TOKEN}"},
        params={"per_page": PER_PAGE, "page": PAGE}
    )
    data = response.json()
    if not data:
        break
    activities.extend(data)
    PAGE += 1

print(f"Total activities fetched: {len(activities)}")

# -------------------------
# PROCESS ACTIVITIES
# -------------------------
activity_rows = []
shoe_ids = set()

for i, act in enumerate(activities, 1):
    gear_id = act.get("gear_id")

    if gear_id:
        shoe_ids.add(gear_id)
            
    distance_m = act.get("distance", 0)
    moving_time_s = act.get("moving_time", 0)
    pace = calculate_pace(distance_m, moving_time_s)

    latlng = act.get("start_latlng")
    if latlng:
        city, country = reverse_lookup(latlng[0], latlng[1])
    else:
        city, country = None, None

    activity_rows.append({
        "activity_id": act.get("id"),
        "start_date": act.get("start_date"),
        "type": act.get("type"),
        "name": act.get("name"),
        "city": city,
        "country": country,
        "distance_km": round(distance_m / 1000, 2),
        "moving_time_min": round(moving_time_s / 60, 1),
        "elevation_gain_m": act.get("total_elevation_gain"),
        "avg_speed_kmh": round(act.get("average_speed", 0) * 3.6, 2),
        "formatted_pace": format_pace(pace),
        "gear_id": gear_id
    })

    if i % 100 == 0:
        print(f"Processed {i}/{len(activities)} activities...")

# ==========================
# DOWNLOAD SHOE DATA
# ==========================

shoe_rows = []


for shoe_id in shoe_ids:
    url = f"https://www.strava.com/api/v3/gear/{shoe_id}"
    response = requests.get(url, headers=headers)
    gear = response.json()

    shoe_rows.append({
        "shoe_id": gear.get("id"),
        "shoe_name": gear.get("name"),
        "total_distance_km": round(gear.get("distance", 0) / 1000, 2),
        "retired": gear.get("retired")
    })

    time.sleep(1)

# -------------------------
# EXPORT TO EXCEL
# -------------------------
activities_df = pd.DataFrame(activity_rows)
shoes_df = pd.DataFrame(shoe_rows)

with pd.ExcelWriter(FILE_NAME) as writer:
    activities_df.to_excel(writer, sheet_name="Activities", index=False)
    shoes_df.to_excel(writer, sheet_name="Shoes", index=False)

print("Strava data fetched successfully")

#finish time
print(f'Process finished at {datetime.now()}')